# Nicheformer — 基础模型教程

**Nicheformer** — 空间感知 Transformer，联合建模空间坐标与基因表达

| 属性 | 值 |
|----------|-------|
| **任务** | embed, integrate, spatial |
| **物种** | human, mouse |
| **基因 ID** | symbol |
| **需要 GPU** | 是 |
| **最低显存** | 16 GB |
| **嵌入维度** | 512 |
| **代码仓库** | [https://github.com/theislab/nicheformer](https://github.com/theislab/nicheformer) |


> **注意：** Nicheformer 具备空间感知能力，最适合用于空间转录组学数据（Visium、MERFISH、Slide-seq）。也可用于解离的单细胞 RNA 测序，但其核心优势在于对空间上下文的建模。

本教程演示如何通过统一的 `ov.fm` API 使用 **Nicheformer**。

**引用：** Zeng, Z. et al. (2024). OmicVerse: a framework for bridging and deepening insights across bulk and single-cell sequencing. *Nature Communications*, 15(1), 5983.

In [ ]:
import omicverse as ov
import scanpy as sc
import os
import warnings
warnings.filterwarnings('ignore')

ov.plot_set()

## 何时选择 Nicheformer 而非标准模型

| 场景 | 推荐模型 |
|----------|------------------|
| 解离单细胞 RNA 测序（无空间信息） | scGPT, Geneformer |
| 空间转录组学（Visium、MERFISH） | **Nicheformer** |
| 跨物种空间比较 | **Nicheformer**（人类 + 小鼠） |
| 微环境/生态位分析 | **Nicheformer** |

Nicheformer 联合建模基因表达与空间坐标，能够捕捉解离模型无法反映的组织组织结构。

## 步骤 1：查看模型规格

使用 `ov.fm.describe_model()` 获取 Nicheformer 的完整规格信息。

In [ ]:
info = ov.fm.describe_model("nicheformer")

print("=== Model Info ===")
print(f"Name: {info['model']['name']}")
print(f"Version: {info['model']['version']}")
print(f"Tasks: {info['model']['tasks']}")
print(f"Species: {info['model']['species']}")
print(f"Embedding dim: {info['model']['embedding_dim']}")
print(f"Differentiator: {info['model']['differentiator']}")

print("\n=== Input Contract ===")
print(f"Gene ID scheme: {info['input_contract']['gene_id_scheme']}")
print(f"Preprocessing: {info['input_contract']['preprocessing']}")

print("\n=== Output Contract ===")
print(f"Embedding key: {info['output_contract']['embedding_key']}")
print(f"Embedding dim: {info['output_contract']['embedding_dim']}")

## 步骤 2：准备数据

加载数据集并将其保存，以供 `ov.fm` 工作流使用。大多数基础模型需要原始计数（非负值）。

In [ ]:
# Nicheformer excels with spatial transcriptomics data.
# For best results, use Visium / MERFISH / Slide-seq data with spatial coordinates.
# Here we demonstrate with PBMC3k (RNA-only) — the model still works for embedding.

adata = sc.datasets.pbmc3k()
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
print(f'Dataset: {adata.n_obs} cells x {adata.n_vars} genes')

# For spatial data, ensure adata.obsm['spatial'] exists:
# adata = sc.read_visium('path/to/visium/')

adata.write_h5ad('pbmc3k_nicheformer.h5ad')


## 步骤 3：分析数据并验证兼容性

在运行推理之前，检查您的数据是否与 Nicheformer 兼容。

In [ ]:
profile = ov.fm.profile_data("pbmc3k_nicheformer.h5ad")

print("=== Data Profile ===")
print(f"Species: {profile['species']}")
print(f"Gene scheme: {profile['gene_scheme']}")
print(f"Modality: {profile['modality']}")
print(f"Cells: {profile['n_cells']:,}")
print(f"Genes: {profile['n_genes']:,}")

# Validate compatibility
validation = ov.fm.preprocess_validate("pbmc3k_nicheformer.h5ad", "nicheformer", "embed")
print(f"\n=== Validation: {validation['status']} ===")
for d in validation.get("diagnostics", []):
    print(f"  [{d['severity']}] {d['message']}")
if validation.get("auto_fixes"):
    print("\nSuggested fixes:")
    for fix in validation["auto_fixes"]:
        print(f"  - {fix}")

## 步骤 4：运行 Nicheformer 推理

通过 `ov.fm.run()` 执行 Nicheformer。该函数负责处理预处理、模型加载、推理和输出写入。

In [ ]:
result = ov.fm.run(
    task="embed",
    model_name="nicheformer",
    adata_path="pbmc3k_nicheformer.h5ad",
    output_path="pbmc3k_nicheformer_out.h5ad",
    device="auto",
)

if "error" in result:
    print(f"Error: {result['error']}")
    if "suggestion" in result:
        print(f"Suggestion: {result['suggestion']}")
else:
    print(f"Status: {result['status']}")
    print(f"Output keys: {result.get('output_keys', [])}")
    print(f"Cells processed: {result.get('n_cells', 0)}")

## 步骤 5：可视化与结果解读

加载输出，从 Nicheformer 嵌入计算 UMAP，并评估质量。

In [ ]:
if os.path.exists("pbmc3k_nicheformer_out.h5ad"):
    adata_out = sc.read_h5ad("pbmc3k_nicheformer_out.h5ad")
    emb_key = "X_nicheformer"
    
    if emb_key in adata_out.obsm:
        print(f"Embedding shape: {adata_out.obsm[emb_key].shape}")
        
        # UMAP visualization
        sc.pp.neighbors(adata_out, use_rep=emb_key)
        sc.tl.umap(adata_out)
        sc.tl.leiden(adata_out, resolution=0.5)
        sc.pl.umap(adata_out, color=["leiden"],
                   title="Nicheformer Embedding (PBMC 3k)")
        
        # QA metrics
        interpretation = ov.fm.interpret_results("pbmc3k_nicheformer_out.h5ad", task="embed")
        if "embeddings" in interpretation["metrics"]:
            for k, v in interpretation["metrics"]["embeddings"].items():
                print(f"\n{k}: dim={v['dim']}", end="")
                if "silhouette" in v:
                    print(f", silhouette={v['silhouette']:.4f}", end="")
                print()
    else:
        print(f"Embedding key {emb_key} not found.")
        print(f"Available keys: {list(adata_out.obsm.keys())}")
else:
    print("Output file not found — check model installation and adapter status.")
    print("See the Guide page for installation instructions.")

## 总结

| 步骤 | 函数 | 功能说明 |
|------|----------|-------------|
| 1 | `ov.fm.describe_model("nicheformer")` | 查看模型规格及输入/输出契约 |
| 2 | `sc.datasets.pbmc3k()` | 准备输入数据 |
| 3 | `ov.fm.profile_data()` + `preprocess_validate()` | 检查兼容性 |
| 4 | `ov.fm.run()` | 执行 Nicheformer 推理 |
| 5 | `ov.fm.interpret_results()` | 评估嵌入质量 |

完整的模型目录请参见 `ov.fm.list_models()` 或 [ov.fm API 概览](t_fm_guide.md)。
Nicheformer 的详细规格说明，请参见 [Nicheformer 指南](t_fm_nicheformer_guide.md)。